# Bài 1.5 · Classification metrics trên dữ liệu thật

**Câu hỏi của buổi này:** khi lớp cần tìm chỉ chiếm 2% dữ liệu, con số nào cho biết model tốt hay tệ, và chọn ngưỡng xác suất thế nào?

Dữ liệu là **Mammography** (OpenML): 11 183 vùng trên ảnh chụp nhũ ảnh, mỗi vùng có 6 đặc trưng đo từ ảnh. 260 vùng có vi vôi hóa; đó là lớp dương cần phát hiện. Model chỉ dùng để tạo xác suất (random forest, học ở bài 2.6); buổi này tập trung đọc kết quả.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng ba tập, biểu đồ hai lớp |
| 2. Bốn ô | confusion matrix của model "luôn đoán âm" và của random forest |
| 3. Precision, recall, F1 | bảng và biểu đồ metric của hai model |
| 4. Ngưỡng | phân bố xác suất hai lớp, precision/recall/F1 theo ngưỡng, ngưỡng chọn trên validation đem lên test |
| 5. ROC và PR | hai đường cong của hai model, AUC và AP |
| 6. Xác suất | log loss, biểu đồ calibration |
| 7. Nhiều lớp | confusion matrix 10 chữ số, F1 từng lớp, macro và weighted, ảnh bị nhầm |
| 8. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **Internet** trong *Session options*; CPU là đủ, chạy hết khoảng 1 phút. Bài giảng: [ai-canuc80k · bài 1.5](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/1.5).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style
    if fmt:
        st = st.format(fmt)
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        yield


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import sklearn
from sklearn.datasets import fetch_openml, load_digits
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, log_loss)

np.random.seed(0)
table(pd.DataFrame([('scikit-learn', sklearn.__version__), ('pandas', pd.__version__), ('numpy', np.__version__)],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

Chia ba tập: **train** để model học, **validation** để chọn ngưỡng, **test** chỉ chấm một lần ở cuối. `stratify` giữ tỉ lệ lớp dương như nhau ở cả ba tập.

In [ ]:
mam = fetch_openml(data_id=310, as_frame=True)
X = mam.data.astype(float).to_numpy()
y = (mam.target.astype(str) == '1').astype(int).to_numpy()

X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=0)

split = pd.DataFrame([(n, len(t), int(t.sum()), t.mean()) for n, t in [('train', y_tr), ('validation', y_va), ('test', y_te)]],
                     columns=['tập', 'số vùng', 'số vùng dương', 'tỉ lệ dương'])
table(split, 'Ba tập, chia giữ tỉ lệ lớp', fmt={'tỉ lệ dương': '{:.2%}'})
check(f'{len(y)} vùng, {y.sum()} vùng dương', len(y) == 11183 and y.sum() == 260)
check('tỉ lệ dương ba tập lệch nhau dưới 0.1 điểm %', np.ptp(split['tỉ lệ dương']) < 0.001)

fig, ax = plt.subplots(figsize=(8, 2.2))
ax.barh(['âm: không vôi hóa', 'dương: có vi vôi hóa'], np.bincount(y), color=[C['gray'], C['clay']])
ax.set_title('Số vùng mỗi lớp'); ax.grid(axis='y', visible=False); bar_labels(ax, '{:.0f}')
plt.tight_layout(); plt.show()
note(f'Cứ khoảng <b>{round(1 / y.mean())}</b> vùng mới có một vùng dương.')

## 2. Bốn ô trước một con số tổng

Quy ước **dương = có vi vôi hóa**. So hai cách đoán trên validation:

- **Luôn đoán âm**: không cần học gì.
- **Random forest**, báo dương khi xác suất từ 0.5 trở lên.

Confusion matrix: hàng là nhãn thật, cột là dự đoán. TN và TP là đoán đúng; FP là báo nhầm; FN là bỏ sót.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=0).fit(X_tr, y_tr)
p_va = rf.predict_proba(X_va)[:, 1]
p_te = rf.predict_proba(X_te)[:, 1]
pred_dummy = np.zeros_like(y_va)
pred_rf = (p_va >= 0.5).astype(int)

def cm_plot(ax, y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    ax.imshow(np.log1p(cm), cmap='Blues', vmin=0, vmax=np.log1p(len(y_true)))
    names = [['TN', 'FP'], ['FN', 'TP']]
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{names[i][j]}\n{cm[i, j]}', ha='center', va='center', fontsize=13,
                    color='white' if cm[i, j] > 200 else '#222')
    ax.set_xticks([0, 1], ['đoán âm', 'đoán dương']); ax.set_yticks([0, 1], ['thật âm', 'thật dương'])
    ax.set_title(title); ax.grid(False)
    return cm

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
cm_d = cm_plot(axes[0], y_va, pred_dummy, 'Luôn đoán âm')
cm_r = cm_plot(axes[1], y_va, pred_rf, 'Random forest, ngưỡng 0.5')
plt.tight_layout(); plt.show()
check('tổng bốn ô bằng số vùng validation', cm_r.sum() == len(y_va) == cm_d.sum())
(tn, fp), (fn, tp) = cm_r
note(f'Luôn đoán âm bỏ sót cả <b>{cm_d[1, 0]}</b> vùng dương. Random forest tìm được <b>{tp}</b> vùng, bỏ sót {fn}, báo nhầm {fp}.')

## 3. Precision, recall và F1

- **Accuracy**: tỉ lệ đoán đúng trên mọi vùng.
- **Precision** $=TP/(TP+FP)$: trong những vùng đã báo dương, bao nhiêu vùng dương thật.
- **Recall** $=TP/(TP+FN)$: trong những vùng dương thật, bao nhiêu vùng được tìm ra.
- **F1** $=2PR/(P+R)$: thấp nếu một trong hai thấp.

In [ ]:
def metrics(y_true, y_pred):
    return {'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'F1': f1_score(y_true, y_pred, zero_division=0)}

m = pd.DataFrame({'luôn đoán âm': metrics(y_va, pred_dummy), 'random forest (0.5)': metrics(y_va, pred_rf)}).T
table(m.reset_index(names='cách đoán'), 'Metric trên validation', fmt={c: '{:.3f}' for c in m.columns}, best='F1')

ax = m.T.plot.barh(figsize=(9, 3.6), color=[C['gray'], C['blue']], width=.75)
ax.set_xlim(0, 1.12); ax.invert_yaxis(); ax.set_title('Accuracy gần bằng nhau, phần còn lại khác hẳn')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1)); bar_labels(ax, '{:.3f}')
plt.tight_layout(); plt.show()
check('F1 khớp công thức 2TP/(2TP+FP+FN)', np.isclose(m.loc['random forest (0.5)', 'F1'], 2 * tp / (2 * tp + fp + fn)))
note(f"Accuracy hai bên chỉ chênh <b>{m['accuracy'].diff().abs().iloc[-1]:.3f}</b>, nhưng recall là 0 so với {m.loc['random forest (0.5)', 'recall']:.2f}.")

## 4. Ngưỡng là một lựa chọn

Báo dương khi xác suất $\ge t$. Hạ $t$ thì báo dương nhiều hơn: recall không giảm, precision thường giảm. Ta thử nhiều ngưỡng **trên validation**, chọn ngưỡng có F1 cao nhất, rồi đem **đúng ngưỡng ấy** lên test một lần.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
bins = np.linspace(0, 1, 21)
ax.hist(p_va[y_va == 0], bins=bins, alpha=.7, color=C['gray'], label=f'thật âm ({(y_va == 0).sum()} vùng)')
ax.hist(p_va[y_va == 1], bins=bins, alpha=.7, color=C['clay'], label=f'thật dương ({(y_va == 1).sum()} vùng)')
ax.axvline(0.5, color='#333', ls='--', lw=1); ax.text(0.51, ax.get_ylim()[1] * .9, 'ngưỡng 0.5', fontsize=9)
ax.set_yscale('log'); ax.set_xlabel('xác suất dương model đưa ra'); ax.set_ylabel('số vùng (thang log)')
ax.set_title('Xác suất của từng vùng validation, tách theo nhãn thật'); ax.legend()
plt.tight_layout(); plt.show()
note(f'{(p_va[y_va == 1] < 0.5).sum()}/{(y_va == 1).sum()} vùng dương có xác suất dưới 0.5: ngưỡng 0.5 bỏ sót chúng.')

In [ ]:
ts = np.round(np.arange(0.05, 0.96, 0.05), 2)
curve = pd.DataFrame([{'ngưỡng': t, **metrics(y_va, (p_va >= t).astype(int)), 'số báo dương': int((p_va >= t).sum())} for t in ts])
t_best = float(curve.loc[curve['F1'].idxmax(), 'ngưỡng'])

fig, ax = plt.subplots(figsize=(9, 4))
for col, c in [('precision', C['blue']), ('recall', C['clay']), ('F1', C['green'])]:
    ax.plot(curve['ngưỡng'], curve[col], marker='o', ms=4, color=c, label=col)
ax.axvline(t_best, color='#333', ls='--', lw=1)
ax.annotate(f'F1 cao nhất ở t = {t_best}', (t_best, curve['F1'].max()), xytext=(10, 18), textcoords='offset points', fontsize=10)
ax.set_xlabel('ngưỡng t'); ax.set_ylim(0, 1.05); ax.set_title('Precision, recall và F1 trên validation khi đổi ngưỡng')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.show()

show = curve[np.isclose(curve['ngưỡng'] * 10 % 1, 0) | (curve['ngưỡng'] == t_best)]
table(show, 'Một số ngưỡng trên validation', fmt={'ngưỡng': '{:.2f}', 'accuracy': '{:.3f}', 'precision': '{:.3f}', 'recall': '{:.3f}', 'F1': '{:.3f}'}, best='F1')
check('recall không tăng khi ngưỡng tăng', (np.diff(curve['recall']) <= 1e-12).all())

In [ ]:
test = pd.DataFrame([{'ngưỡng': name, **metrics(y_te, (p_te >= t).astype(int)), 'số báo dương': int((p_te >= t).sum())}
                     for name, t in [('0.5 (mặc định)', 0.5), (f'{t_best} (chọn trên validation)', t_best)]])
table(test, f'Test, chấm một lần ({len(y_te)} vùng, {y_te.sum()} vùng dương)', fmt={c: '{:.3f}' for c in ['accuracy', 'precision', 'recall', 'F1']}, best='F1')
f_va = curve.loc[curve['F1'].idxmax(), 'F1']
f_te05, f_teb = test['F1'].iloc[0], test['F1'].iloc[1]
msg = (f"Ngưỡng {t_best}: F1 validation {f_va:.3f}, F1 test <b>{f_teb:.3f}</b>. ")
if f_teb < f_te05:
    msg += (f"Lần này ngưỡng 0.5 lại cao hơn trên test ({f_te05:.3f}): validation chỉ có {(y_va == 1).sum()} vùng dương, nên ngưỡng chọn trên đó còn dao động. "
            "Dù vậy vẫn giữ ngưỡng đã chốt; đổi theo test là dùng test để chọn.")
else:
    msg += f"Cao hơn ngưỡng 0.5 ({f_te05:.3f}) trên test: ngưỡng chọn trên validation dùng được cho dữ liệu mới."
note(msg)

## 5. ROC và PR: thứ tự trước khi cắt ngưỡng

Thêm một model thứ hai, logistic regression, để so. Hai đường cong quét mọi ngưỡng cùng lúc:

- **ROC**: trục ngang FPR $=FP/(FP+TN)$, trục dọc TPR = recall. Diện tích dưới đường là **ROC-AUC**; đoán bừa được 0.5.
- **PR**: trục ngang recall, trục dọc precision. Diện tích tóm bằng **AP** (average precision); đoán bừa được đúng tỉ lệ dương.

In [ ]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
probs = {'random forest': p_te, 'logistic regression': lr.predict_proba(X_te)[:, 1]}
prev = y_te.mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
rows = []
for (name, p), c in zip(probs.items(), [C['blue'], C['sand']]):
    fpr, tpr, _ = roc_curve(y_te, p); auc = roc_auc_score(y_te, p)
    rec, prec, _ = precision_recall_curve(y_te, p); ap = average_precision_score(y_te, p)
    axes[0].plot(fpr, tpr, color=c, lw=2, label=f'{name}: AUC {auc:.3f}')
    axes[1].plot(rec, prec, color=c, lw=2, label=f'{name}: AP {ap:.3f}')
    rows.append((name, auc, ap))
axes[0].plot([0, 1], [0, 1], color=C['gray'], ls='--', label='đoán bừa: 0.5')
axes[1].axhline(prev, color=C['gray'], ls='--', label=f'đoán bừa: {prev:.3f}')
axes[0].set(xlabel='FPR (tỉ lệ vùng âm bị báo nhầm)', ylabel='TPR = recall', title='ROC')
axes[1].set(xlabel='recall', ylabel='precision', title='Precision–recall', ylim=(0, 1.02))
for ax in axes: ax.legend(loc='lower right' if ax is axes[0] else 'upper right')
plt.tight_layout(); plt.show()

curves = pd.DataFrame(rows, columns=['model', 'ROC-AUC', 'AP'])
table(curves, 'Diện tích dưới hai đường cong (test)', fmt={'ROC-AUC': '{:.3f}', 'AP': '{:.3f}'}, best='AP')
check('cả hai model vượt mức đoán bừa', (curves['ROC-AUC'] > 0.5).all() and (curves['AP'] > prev).all())

In [ ]:
fpr, tpr, thr = roc_curve(y_te, p_te)
k = int(np.argmax(tpr >= 0.8))
n_neg, n_pos = (y_te == 0).sum(), (y_te == 1).sum()
note(f'Trên ROC của random forest, điểm recall {tpr[k]:.2f} có FPR chỉ <b>{fpr[k]:.3f}</b>. Nhưng vì có {n_neg} vùng âm, đó là '
     f'<b>{round(fpr[k] * n_neg)}</b> vùng báo nhầm, so với {round(tpr[k] * n_pos)} vùng tìm đúng. Với lớp hiếm, ROC trông đẹp hơn trải nghiệm thật; đường PR cho thấy điều này rõ hơn.')

## 6. Xác suất: log loss và calibration

**Log loss** chấm chính xác suất, không chỉ nhãn: đoán sai mà rất tự tin thì bị phạt nặng. **Calibration** hỏi: chia các vùng thành nhóm theo xác suất model đưa ra; trong nhóm model nói khoảng 0.3, có khoảng 30% là dương thật không? Mốc so là một hằng số: mọi vùng đều nhận xác suất bằng tỉ lệ dương của train.

In [ ]:
const = np.full(len(y_te), y_tr.mean())
ll = pd.DataFrame([(name, log_loss(y_te, np.clip(p, 1e-6, 1 - 1e-6)), p.mean()) for name, p in [*probs.items(), ('hằng số = tỉ lệ dương', const)]],
                  columns=['model', 'log loss', 'xác suất trung bình'])
ll['tỉ lệ dương thật'] = prev
table(ll, 'Log loss trên test (thấp hơn là tốt hơn)', fmt={'log loss': '{:.4f}', 'xác suất trung bình': '{:.4f}', 'tỉ lệ dương thật': '{:.4f}'}, best='log loss', lower_is_better=True)

edges = [0, 0.02, 0.1, 0.3, 0.6, 1.0001]
names = ['< 0.02', '0.02–0.1', '0.1–0.3', '0.3–0.6', '≥ 0.6']
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, (name, p) in zip(axes, probs.items()):
    g = pd.DataFrame({'nhóm': pd.cut(p, edges, right=False, labels=names), 'p': p, 'y': y_te}).groupby('nhóm', observed=False)
    cal = pd.DataFrame({'số vùng': g.size(), 'model nói': g['p'].mean(), 'thực tế': g['y'].mean()}).fillna(0)
    xs = np.arange(len(cal))
    ax.bar(xs - .2, cal['model nói'], .4, color=C['blue'], label='model nói (xác suất trung bình)')
    ax.bar(xs + .2, cal['thực tế'], .4, color=C['clay'], label='thực tế (tỉ lệ dương)')
    bar_labels(ax, '{:.2f}')
    ax.set_xticks(xs, [f'{n}\n{k} vùng' for n, k in zip(cal.index, cal['số vùng'])])
    ax.set_title(name); ax.set_ylim(0, 1.05); ax.grid(axis='x', visible=False)
axes[0].set_ylabel('xác suất / tỉ lệ'); axes[0].legend(loc='upper left')
fig.suptitle('Calibration: hai cột cao gần bằng nhau là xác suất đáng tin', y=1.02)
plt.tight_layout(); plt.show()
check('log loss của model thấp hơn hằng số', (ll['log loss'].iloc[:2] < ll['log loss'].iloc[2]).all())
note(f"Hằng số có log loss {ll['log loss'].iloc[2]:.4f}; random forest {ll['log loss'].iloc[0]:.4f}, logistic {ll['log loss'].iloc[1]:.4f}. "
     f"Cả hai đều thấp hơn hằng số: xác suất của model mang thêm thông tin, không chỉ nhãn.")

## 7. Nhiều lớp

Bộ **digits** của scikit-learn: 1 797 ảnh chữ số viết tay 8×8, 10 lớp. Với mỗi lớp, coi lớp ấy là dương rồi tính precision, recall, F1. Trên confusion matrix (hàng là nhãn thật), recall của một lớp đọc theo hàng, precision đọc theo cột.

- **Macro-F1**: trung bình đều F1 của các lớp.
- **Weighted-F1**: trung bình theo số mẫu của từng lớp.
- **Micro-F1**: gộp mọi ô rồi tính; với bài một nhãn, nó bằng accuracy.

In [ ]:
dg = load_digits()
Xd_tr, Xd_te, yd_tr, yd_te, img_tr, img_te = train_test_split(dg.data / 16, dg.target, dg.images, test_size=0.3, stratify=dg.target, random_state=0)
clf = LogisticRegression(max_iter=2000, C=0.05).fit(Xd_tr, yd_tr)
yd_pred = clf.predict(Xd_te)
cm = confusion_matrix(yd_te, yd_pred)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(cm, cmap='Blues'); ax.grid(False)
for i in range(10):
    for j in range(10):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=9, color='white' if cm[i, j] > cm.max() / 2 else ('#c2603e' if i != j else '#222'))
ax.set_xticks(range(10)); ax.set_yticks(range(10)); ax.set_xlabel('dự đoán'); ax.set_ylabel('nhãn thật')
ax.set_title(f'Confusion matrix trên {len(yd_te)} ảnh test (ô đỏ là nhầm)')
plt.tight_layout(); plt.show()

per = pd.DataFrame({'chữ số': range(10),
                    'precision': precision_score(yd_te, yd_pred, average=None),
                    'recall': recall_score(yd_te, yd_pred, average=None),
                    'F1': f1_score(yd_te, yd_pred, average=None),
                    'số ảnh': np.bincount(yd_te)})
off = cm - np.diag(np.diag(cm)); a, b = np.unravel_index(off.argmax(), off.shape)
worst = per.loc[per['F1'].idxmin()]
note(f"Cặp nhầm nhiều nhất: số <b>{a}</b> bị đoán thành <b>{b}</b> ({off[a, b]} ảnh). Lớp có F1 thấp nhất: số <b>{int(worst['chữ số'])}</b> ({worst['F1']:.3f}).")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(per['chữ số'].astype(str), per['F1'], color=[C['clay'] if v == per['F1'].min() else C['blue'] for v in per['F1']])
ax.set_ylim(0.8, 1.03); ax.set_xlabel('chữ số'); ax.set_ylabel('F1'); ax.set_title('F1 từng lớp'); bar_labels(ax, '{:.2f}')
plt.tight_layout(); plt.show()

avg = pd.DataFrame([('macro-F1', f1_score(yd_te, yd_pred, average='macro')), ('weighted-F1', f1_score(yd_te, yd_pred, average='weighted')),
                    ('micro-F1', f1_score(yd_te, yd_pred, average='micro')), ('accuracy', accuracy_score(yd_te, yd_pred))],
                   columns=['cách tính', 'giá trị'])
table(avg, 'Bốn con số tổng cho 10 lớp', fmt={'giá trị': '{:.4f}'})
check('micro-F1 bằng accuracy', np.isclose(avg['giá trị'].iloc[2], avg['giá trị'].iloc[3]))
check('macro-F1 bằng trung bình cộng F1 từng lớp', np.isclose(avg['giá trị'].iloc[0], per['F1'].mean()))

wrong = np.flatnonzero(yd_pred != yd_te)[:8]
fig, axes = plt.subplots(1, len(wrong), figsize=(1.6 * len(wrong), 2.2))
for ax, i in zip(np.atleast_1d(axes), wrong):
    ax.imshow(img_te[i], cmap='gray_r'); ax.axis('off')
    ax.set_title(f'thật {yd_te[i]}\nđoán {yd_pred[i]}', fontsize=10, color=C['clay'])
fig.suptitle('Những ảnh bị đoán sai', y=1.05)
plt.tight_layout(); plt.show()

Digits có 10 lớp gần đều nhau, nên macro và weighted gần bằng nhau. Quay lại Mammography, nơi một lớp chiếm 98%: hai cách lấy trung bình cho hai câu chuyện khác hẳn.

In [ ]:
def avg_row(name, pred):
    return {'cách đoán': name, 'F1 lớp dương': f1_score(y_te, pred, zero_division=0), 'F1 lớp âm': f1_score(y_te, pred, pos_label=0),
            'macro-F1': f1_score(y_te, pred, average='macro', zero_division=0), 'weighted-F1': f1_score(y_te, pred, average='weighted', zero_division=0),
            'accuracy': accuracy_score(y_te, pred)}
bal = pd.DataFrame([avg_row('luôn đoán âm', np.zeros_like(y_te)), avg_row(f'random forest, ngưỡng {t_best}', (p_te >= t_best).astype(int))])
table(bal, 'Mammography, test: macro và weighted khi lớp lệch', fmt={c: '{:.3f}' for c in bal.columns[1:]}, best='macro-F1')
note(f"Luôn đoán âm vẫn có weighted-F1 <b>{bal['weighted-F1'].iloc[0]:.3f}</b> vì lớp âm chiếm gần hết trọng số; macro-F1 chỉ {bal['macro-F1'].iloc[0]:.3f}. "
     f"Đề chấm macro-F1 là đang bắt model phải tìm lớp hiếm.")

## 8. Tóm tắt

In [ ]:
summary = [('tỉ lệ vùng dương', f'{y.mean():.2%}'),
           ('accuracy của "luôn đoán âm" (validation)', f"{m.loc['luôn đoán âm', 'accuracy']:.3f}"),
           ('F1 random forest, ngưỡng 0.5 (test)', f"{test['F1'].iloc[0]:.3f}"),
           (f'F1 random forest, ngưỡng {t_best} chọn trên validation (test)', f"{test['F1'].iloc[1]:.3f}"),
           ('ROC-AUC / AP random forest (test)', f"{curves['ROC-AUC'].iloc[0]:.3f} / {curves['AP'].iloc[0]:.3f}"),
           ('macro-F1 trên 10 chữ số', f"{avg['giá trị'].iloc[0]:.3f}")]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** thêm `class_weight='balanced'` cho random forest rồi xem confusion matrix đổi thế nào; chọn ngưỡng theo điều kiện "recall ít nhất 0.8" thay vì F1 cao nhất; đổi `strategy='uniform'` ở biểu đồ calibration.